In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# One-hot representation (Isolated)

king_oh = np.array([1, 0, 0])
queen_oh = np.array([0, 1, 0])
apple_oh = np.array([0, 0, 1])

# Embedding representation (Semantic)

king_emb = np.array([0.99, 0.95, 0.01])
queen_emb = np.array([0.90, 0.96, 0.02])
apple_emb = np.array([0.01, 0.01, 0.98])

print(f"One-Hot Similarity (King vs Queen): {cosine_similarity([king_oh], [queen_oh])[0][0]}")
print(f"Embedding Similarity (King vs Queen): {cosine_similarity([king_emb], [queen_emb])[0][0]:.4f}")

In [ ]:
pip install gensim

In [ ]:
from gensim.models import Word2Vec
import pandas as pd

sentences = [
    ["the", "siren", "wails", "loudly"],
    ["emergency", "vehicles", "have", "a", "siren"],
    ["the", "dog", "will", "bark", "at", "strangers"],
    ["a", "loud", "bark", "woke", "me", "up"]
]

# 1. Train CBOW (sg=0)
model_cbow = Word2Vec(sentences, vector_size=10, window=2, sg=0, min_count=1, epochs=100)

# 2. Train Skip-gram (sg=1)
model_sg = Word2Vec(sentences, vector_size=10, window=2, sg=1, min_count=1, epochs=100)

# Function to compare results
def compare_word(word):
    cbow_sim = model_cbow.wv.most_similar(word, topn=2)
    sg_sim = model_sg.wv.most_similar(word, topn=2)
    return cbow_sim, sg_sim

# Compare for our target sound classes
for word in ['siren', 'bark']:
    cbow, sg = compare_word(word)
    print(f"\nResults for '{word}':")
    print(f"  CBOW (predicts word from context): {cbow}")
    print(f"  Skip-gram (predicts context from word): {sg}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE


words = list(model_cbow.wv.index_to_key)
vectors = model_cbow.wv[words]


tsne = TSNE(n_components=2, random_state=42, perplexity=len(words)-1)
reduced_vectors = tsne.fit_transform(vectors)

plt.figure(figsize=(8, 6))
for i, word in enumerate(words):
    plt.scatter(reduced_vectors[i, 0], reduced_vectors[i, 1])
    plt.annotate(word, (reduced_vectors[i, 0], reduced_vectors[i, 1]))
plt.title("Language Latent Space: Word Clusters")
plt.show()

In [ ]:
import gensim.downloader as api

# Load a small pre-trained model (Twitter 25-dimensional vectors)
glove_model = api.load("glove-twitter-25")

# Semantic Math: King - Man + Woman = ?
result = glove_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)
print(f"King - Man + Woman = {result[0][0]}")

# Finding the 'Odd One Out'
print("Which doesn't fit? [apple, banana, siren]:", glove_model.doesnt_match(["apple", "banana", "siren"]))

In [ ]:
from transformers import AutoTokenizer

# 1. WordPiece (BERT)
tokenizer_wp = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Byte-level BPE (GPT-2)
tokenizer_bbpe = AutoTokenizer.from_pretrained("gpt2")

text = "Sirens are wailing loudly."

tokens_wp = tokenizer_wp.tokenize(text)
tokens_bbpe = tokenizer_bbpe.tokenize(text)

print(f"WordPiece (BERT): {tokens_wp}")
print(f"Byte-level BPE (GPT-2): {tokens_bbpe}")